In [0]:
%run ./ukey_match_regular_common_business

#main

In [0]:
def check_uid_duplicate(task_id, logger):
    # 1. check
    ukey_with_flag_df = get_ukeyGroup_with_flag(task_id)

    # 2. modify log content
    masterUkey_duplicate_records = [
        row[0] 
        for row in ukey_with_flag_df
            .filter(F.col("is_master_recode") == True)
            .filter(F.col("is_masterUkey_duplicate") == True)
            .select("matc_id")
            .collect()
    ]

    if len(masterUkey_duplicate_records) > 0:
        masterUkey_duplicate_records_str = ",".join([f"'{item}'" for item in masterUkey_duplicate_records])
        logger.status = "WARNING"
        logger.message = f"The following records have multiple new_comsumermdmkeys corresponding to the same master_comsumermdmkey, matc_id is: {masterUkey_duplicate_records_str}"

    cid_duplicate_records = [
        row[0]
        for row in ukey_with_flag_df
            .filter(F.col("is_cid_duplicate") == True)
            .select("matc_id")
            .collect()
    ]

    if len(cid_duplicate_records) > 0:
        cid_duplicate_records_str = ",".join([f"'{item}'" for item in cid_duplicate_records])
        if logger.status == "WARNING":
            logger.message =  f"{logger.message} \nThe following records have multiple new_comsumermdmkeys corresponding to the same mrkt_comde, brnd_comde, source_comde, and consumer_id, matc_id is: {cid_duplicate_records_str}"
        else:
            logger.status = "WARNING"
            logger.message = f"The following records have multiple new_comsumermdmkeys corresponding to the same mrkt_comde, brnd_comde, source_comde, and consumer_id, matc_id is: {cid_duplicate_records_str}"
     

    # 3. mark exclude data
    update_exclude_df = (ukey_with_flag_df
            .filter(F.col("is_master_recode") == False)
            .filter(F.col("is_cid_duplicate") == True)
            .select("task_id", "mrkt_code", "srcc_id")
            .distinct()
    )

    if update_exclude_df.count() > 0:
        clean_consumer_delta_table = DeltaTable.forName(spark, f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_consumer")
        
        (clean_consumer_delta_table
            .alias("target")
            .merge(
                update_exclude_df.alias("source"),
                "target.TASK_ID = source.task_id AND "
                "target.SRCC_MRKT_CODE = source.mrkt_code AND "
                "target.SRCC_ID = source.srcc_id "
            )
            .whenMatchedUpdate(
                set={
                    "IS_INCLUDE": F.lit(False),
                    "EXCLUDE_TYPE": F.lit(EXCLUDE_TYPE_BY_UID_DUPLICATE)
                }
            )
            .execute()
        )

In [0]:
def ukey_match_regular_process(task_id, logger):
    t_merge_exclude_phone_config, t_merge_exclude_media_config, t_merge_exclude_address_config, t_merge_exclude_consumer_config, exclude_phones, exclude_emails = get_exclude_dfs()

    # 1.1 batch data
    print_log("1.1 batch data")
    output1_df, t_clean_consumer, clean_phone_vld, clean_emedia_vld, clean_address_all = get_batch_data(task_id, t_merge_exclude_consumer_config, t_merge_exclude_phone_config, t_merge_exclude_media_config, t_merge_exclude_address_config)
    
    # output1_df.cache()
    # print_log(f"output1_df count: {output1_df.count()}")

    t_clean_consumer.cache()
    print_log(f"t_clean_consumer count: {t_clean_consumer.count()}")

    clean_phone_vld.cache()
    print_log(f"clean_phone_vld count: {clean_phone_vld.count()}")

    clean_emedia_vld.cache()
    print_log(f"clean_emedia_vld count: {clean_emedia_vld.count()}")

    clean_address_all.cache()
    print_log(f"clean_address_all count: {clean_address_all.count()}")

    # 1.2 master data
    print_log("1.2 master data")
    output2_df = get_master_data(t_merge_exclude_consumer_config, exclude_phones, exclude_emails, t_merge_exclude_address_config, t_clean_consumer, clean_phone_vld, clean_emedia_vld, clean_address_all)

    # output2_df.cache()
    # print_log(f"output2_df count: {output2_df.count()}")

    # 1.3 batch union master, generate match key
    merged_df = generate_match_key(output1_df.unionByName(output2_df))

    # 2.1 generate vertices, edges
    print_log("2.1 generate vertices, edges")
    df_with_id = merged_df.withColumn("id", F.concat_ws("_", F.col("scon_mrkt_code"), F.col("scon_srcc_id")))

    df_with_id.cache()
    print_log(f"df_with_id count: {df_with_id.count()}")

    vertices, edges = generate_vertices_and_edge(df_with_id)
    edges.cache()
    print_log(f"edges count: {edges.count()}")

    print_log(f"save start : {get_env_config('silver_consumer_cleansed_database')}.t_clean_ukey_edge")
    save_to_target_table(
        edges.withColumn("task_id", F.lit(task_id)).withColumn("creation_dt", F.current_timestamp()),
        f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_ukey_edge",
        f"task_id = '{task_id}' "
    )
    print_log(f"save end : {get_env_config('silver_consumer_cleansed_database')}.t_clean_ukey_edge")
    
    # 2.2 generate gid
    print_log("2.2 generate_gid")
    final_df = generate_gid(vertices, edges, df_with_id)

    # final_df.cache()
    # print_log(f"final_df count: {final_df.count()}")

    # 3. new Ukey generate
    print_log("3. new Ukey generate")
    final_with_newUkey_df = generate_new_ukey(final_df)
    
    print_log(f"save start : {get_env_config('silver_consumer_cleansed_database')}.t_clean_ukey_group")
    save_to_target_table(
        final_with_newUkey_df,
        f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_ukey_group",
        f"task_id = '{task_id}' and match_type = '{MATCH_TYPE_REGULAR_STR}' "
    )
    print_log(f"save end : {get_env_config('silver_consumer_cleansed_database')}.t_clean_ukey_group")
    
    t_clean_consumer.unpersist()
    clean_phone_vld.unpersist()
    clean_emedia_vld.unpersist()
    clean_address_all.unpersist()
    # output1_df.unpersist()
    # output2_df.unpersist()
    df_with_id.unpersist()
    edges.unpersist()
    # final_df.unpersist()
    

    # 4. check uid duplicate
    check_uid_duplicate(task_id, logger)

In [0]:
task_id = dbutils.widgets.get("task_id")
print(f"task_id: {task_id}")

with StepLogger("4.1_ukey_match_regular", "04-1", "consumerlist", task_id=task_id) as logger:
    ukey_match_regular_process(task_id, logger)